# Pure Functions

Goal: browse the different kinds of pure and impure functions in Python — what makes a function pure (same input -> same output, no side effects), the common ways functions become impure (global state, mutation, I/O, randomness/time), referential transparency, and why purity matters for composition, caching, and testing.

## 1. A basic pure function

A pure function's return value depends only on its arguments, and calling it never changes anything outside itself. Calling it repeatedly with the same inputs always produces the same output.

In [1]:
def add(a, b):
    return a + b

print(add(2, 3))
print(add(2, 3))  # same inputs -> same output, every time
print(add(10, -4))

5
5
6


## 2. Impure: depends on / mutates global state

`add_impure` reads and mutates a variable outside its own scope. Same inputs, different outputs on each call — that's a side effect, and it breaks purity.

In [2]:
counter = 0

def add_impure(a, b):
    global counter
    counter += 1  # side effect: mutates state outside the function
    return a + b + counter

print(add_impure(2, 3))
print(add_impure(2, 3))  # same inputs, different output -> not pure

6
7


## 3. Impure: mutating a mutable argument

Mutating an argument that was passed in (like a list) is a side effect visible to the caller. The pure version instead builds and returns a new object, leaving the original untouched.

In [3]:
def append_impure(lst, item):
    lst.append(item)  # mutates the caller's list -> side effect
    return lst

def append_pure(lst, item):
    return lst + [item]  # builds and returns a new list, leaves input untouched

original = [1, 2, 3]
print(append_impure(original, 4))
print(original)  # original was mutated

original2 = [1, 2, 3]
print(append_pure(original2, 4))
print(original2)  # original2 is untouched

[1, 2, 3, 4]
[1, 2, 3, 4]
[1, 2, 3, 4]
[1, 2, 3]


## 4. Impure: I/O (printing, files, network)

Any function that performs I/O has a side effect the outside world can observe. Returning a value instead of printing it keeps the function pure and lets the caller decide what to do with the result.

In [4]:
def greet_impure(name):
    print(f"Hello, {name}!")  # side effect: writes to stdout, returns nothing useful

def greet_pure(name):
    return f"Hello, {name}!"  # no side effect, caller decides what to do with it

greet_impure("Alice")

message = greet_pure("Bob")
print(message)

Hello, Alice!
Hello, Bob!


## 5. Impure: randomness and time

`random.randint` (and anything reading the system clock) pulls from hidden state outside the function's arguments, so the same call can return different results. Passing the source of randomness in explicitly (e.g. a seed) restores determinism.

In [5]:
import random

def roll_impure():
    return random.randint(1, 6)  # depends on hidden global state -> not pure

def roll_pure(seed):
    return random.Random(seed).randint(1, 6)  # deterministic: seed fully determines output

print(roll_pure(42))
print(roll_pure(42))  # same seed -> same result every time
print(roll_pure(7))

6
6
3


## 6. Referential transparency

Because a pure function always returns the same value for the same arguments, any call to it can be replaced by its result without changing the program's behavior.

In [6]:
def square(n):
    return n * n

# referential transparency: square(4) can be replaced by 16 anywhere
# without changing the meaning of the program
a = square(4) + square(4)
b = 16 + 16
print(a == b)  # True
print(a, b)

True
32 32


## 7. Composing pure functions with `map`, `filter`, `reduce`

Pure functions compose safely — chaining them together never has to worry about hidden state or execution order affecting the result.

In [7]:
from functools import reduce

def double(n):
    return n * 2

def is_even(n):
    return n % 2 == 0

def multiply(a, b):
    return a * b

numbers = [1, 2, 3, 4, 5, 6]

doubled = list(map(double, numbers))
print(doubled)  # [2, 4, 6, 8, 10, 12]

evens = list(filter(is_even, numbers))
print(evens)  # [2, 4, 6]

product = reduce(multiply, numbers)
print(product)  # 720

[2, 4, 6, 8, 10, 12]
[2, 4, 6]
720


## 8. Purity enables safe caching

Because a pure function's output only depends on its input, its results can be memoized — `functools.lru_cache` is safe to use here. Caching an impure function would silently return stale results.

In [8]:
from functools import lru_cache
import time

@lru_cache(maxsize=None)
def fib_pure(n):
    if n < 2:
        return n
    return fib_pure(n - 1) + fib_pure(n - 2)  # deterministic -> safe to cache

start = time.perf_counter()
print(fib_pure(30))
first_call = time.perf_counter() - start

start = time.perf_counter()
print(fib_pure(30))  # served from cache, no recomputation
cached_call = time.perf_counter() - start

print(cached_call < first_call)

832040
832040
True


## 9. Purity makes testing trivial

No setup, no mocks, no teardown — just call the function and check the output. This is one of the biggest practical payoffs of writing pure functions.

In [9]:
def add(a, b):
    return a + b

# pure functions are trivial to test: no setup, no mocking, no teardown
assert add(2, 3) == 5
assert add(-1, 1) == 0
assert add(0, 0) == 0
print("all assertions passed")

all assertions passed
